# Custom Authentication with Request Lambda Interceptor

## Introduction

When deploying AI agents with Amazon Bedrock AgentCore, organizations often need to integrate with enterprise APIs that use authentication methods not natively supported by the Gateway. The Gateway provides native support for OAuth 2.0, AWS IAM, and API key authentication — but many enterprise APIs still rely on other mechanisms such as HTTP Basic Authentication (RFC 7617) and HMAC-based request signing (RFC 2104).

This tutorial shows how to use the **Request Lambda Interceptor** to implement custom authentication for downstream tool APIs. You'll explore two patterns:

1. **JWT-to-Basic-Auth credential translation** — Decodes the inbound JWT, looks up per-user credentials in AWS Secrets Manager, and replaces the Bearer token with a Basic Auth header.
2. **HMAC-based request signing** — Generates a time-limited HMAC-SHA256 signature over request metadata for high-security APIs.

Both patterns run through the same gateway — the interceptor routes to the correct handler based on the tool name in the MCP request body.

### Tutorial Details

| Information | Details |
| :--- | :--- |
| Tutorial type | Interactive |
| AgentCore components | AgentCore Gateway |
| Gateway Target type | AWS Lambda |
| Inbound Auth IdP | Amazon Cognito (adaptable to any OIDC provider) |
| Outbound Auth | API Key (placeholder) + Request Lambda Interceptor |
| Example complexity | Intermediate |
| SDK used | boto3 |

### Prerequisites

Before running this tutorial, you should have:
- An existing AgentCore Gateway with inbound JWT authentication configured
- Per-user Basic Auth credentials stored in AWS Secrets Manager (convention: `<prefix>/<user-pool-id>/<email>`)
- An HMAC shared secret stored in Secrets Manager
- An IAM role for the interceptor Lambda with `secretsmanager:GetSecretValue` permissions

**Cost considerations:** This tutorial creates AWS resources that incur charges, including Lambda functions, IAM roles, gateway targets, and API key credential providers. Follow the Cleanup steps at the end to delete these resources and avoid ongoing costs.

---

## Step 1: Install Dependencies

Install the required packages for interacting with AWS services.

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'boto3'])
print('Dependencies installed successfully')

## Step 2: Setup & Configuration

Initialize AWS clients and set configuration variables. Replace the placeholder values with your own.

In [ ]:
import boto3
import json
import time
import zipfile
import io
from datetime import datetime
from botocore.exceptions import ClientError

# ── Replace these with your values ──────────────────────────────────────
region = 'us-east-1'                          # Your AWS region
gateway_id = '<your-gateway-id>'              # Your existing AgentCore Gateway ID
user_pool_id = '<your-cognito-user-pool-id>'  # Your Cognito User Pool ID
hmac_secret_name = '<your-hmac-secret-name>'  # Name of your HMAC secret in Secrets Manager
# ───────────────────────────────────────────────────────────────────────

timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')

lambda_client = boto3.client('lambda', region_name=region)
iam_client = boto3.client('iam', region_name=region)
agentcore_client = boto3.client('bedrock-agentcore-control', region_name=region)
sts_client = boto3.client('sts', region_name=region)
account_id = sts_client.get_caller_identity()['Account']

print(f'Region:    {region}')
print(f'Account:   {account_id}')
print(f'Gateway:   {gateway_id}')
print(f'Timestamp: {timestamp}')

## Step 3: Deploy the Request Lambda Interceptor

This is the core of the tutorial. The interceptor Lambda sits between the gateway and your downstream tools. When a request arrives, it:

1. Inspects the **tool name** from the MCP request body (`body.params.name`)
2. Routes to the appropriate authentication handler:
   - **JWT-to-Basic-Auth** — Decodes the JWT, extracts user identity from the `onBehalfOf` claim, looks up per-user credentials in Secrets Manager, and replaces the `Authorization` header with Basic Auth (RFC 7617)
   - **HMAC signing** — Generates an HMAC-SHA256 signature over the request method, path, timestamp, and gateway host, then injects `X-Signature`, `X-Timestamp`, and `X-Gateway-Host` headers
3. Returns the transformed request to the gateway for forwarding

The following cell defines the full interceptor code and deploys it as a Lambda function with the required IAM role and Secrets Manager permissions.

In [ ]:
# ---------------------------------------------------------------------------
# Full Request Lambda Interceptor code
# ---------------------------------------------------------------------------
interceptor_code = '''
import json
import logging
import hmac
import hashlib
import time
import base64
import os
import boto3

logger = logging.getLogger()
logger.setLevel(logging.INFO)

secrets_client = boto3.client("secretsmanager")
USER_POOL_ID = os.environ["USER_POOL_ID"]
HMAC_SECRET_NAME = os.environ["HMAC_SECRET_NAME"]
HMAC_SECRET = None
HMAC_SECRET_LOADED_AT = 0
HMAC_SECRET_TTL = 300  # Refresh cached secret every 5 minutes


# ===========================================================================
# Secret retrieval helpers
# ===========================================================================

def get_hmac_secret():
    """Retrieve the HMAC shared secret from Secrets Manager (cached with TTL)."""
    global HMAC_SECRET, HMAC_SECRET_LOADED_AT
    if HMAC_SECRET is None or (time.time() - HMAC_SECRET_LOADED_AT) > HMAC_SECRET_TTL:
        try:
            response = secrets_client.get_secret_value(SecretId=HMAC_SECRET_NAME)
            HMAC_SECRET = response["SecretString"]
            HMAC_SECRET_LOADED_AT = time.time()
            logger.info("HMAC shared secret loaded from Secrets Manager.")
        except (secrets_client.exceptions.ResourceNotFoundException, secrets_client.exceptions.InvalidRequestException, ConnectionError) as e:
            logger.error(f"Failed to retrieve HMAC secret: {e}")
            raise RuntimeError("HMAC secret not available.")
    return HMAC_SECRET


def get_basic_auth_credentials(email):
    """Look up per-user Basic Auth credentials from Secrets Manager.

    Uses the naming convention: basic-auth-demo/<pool-id>/<email>
    Adapt this pattern to match your organization's secret naming convention.
    """
    secret_name = f"basic-auth-demo/{USER_POOL_ID}/{email}"  # Replace with your secret naming convention
    try:
        response = secrets_client.get_secret_value(SecretId=secret_name)
        creds = json.loads(response["SecretString"])
        logger.info(f"[BasicAuth] Retrieved credentials (role: {creds.get('role', 'unknown')}).")
        return creds
    except (secrets_client.exceptions.ResourceNotFoundException, secrets_client.exceptions.InvalidRequestException, ConnectionError) as e:
        logger.error(f"[BasicAuth] Failed to retrieve credentials: {e}")
        return None


# ===========================================================================
# JWT decoding
# ===========================================================================

def decode_jwt(token):
    """Decode JWT payload to extract claims.

    The Gateway has already verified the JWT signature — the interceptor
    only decodes the payload to extract claims for credential lookup.
    """
    try:
        payload = token.split(".")[1]
        payload += "=" * (4 - len(payload) % 4)
        decoded = base64.urlsafe_b64decode(payload)
        claims = json.loads(decoded)
        logger.info(f"[JWT] Decoded claims (sub: {claims.get('sub')}).")
        return claims
    except (IndexError, ValueError, json.JSONDecodeError) as e:
        logger.error(f"[JWT] Failed to decode token: {e}")
        return None


# ===========================================================================
# Pattern 1: JWT-to-Basic-Auth translation
# ===========================================================================

def build_basic_auth_header(headers):
    """Decode the inbound JWT, extract user identity from the onBehalfOf
    claim, look up per-user credentials in Secrets Manager, and replace
    the Bearer token with a Basic Auth header (RFC 7617)."""
    logger.info("[BasicAuth] Building Basic Authentication header.")

    auth_header = headers.get("Authorization", "")
    if not auth_header.startswith("Bearer "):
        return _error_response(401, "No Bearer token found in request.")

    claims = decode_jwt(auth_header[7:])
    if not claims:
        return _error_response(401, "Failed to decode JWT token.")

    # Extract user identity from onBehalfOf claim (agent token exchange)
    on_behalf_of = claims.get("onBehalfOf")
    if not on_behalf_of:
        return _error_response(401, "Missing onBehalfOf claim. Agent must use token exchange.")

    if isinstance(on_behalf_of, str):
        try:
            on_behalf_of = json.loads(on_behalf_of)
        except (json.JSONDecodeError, TypeError):
            return _error_response(401, "Malformed onBehalfOf claim.")

    email = on_behalf_of.get("email") or on_behalf_of.get("username")

    # Validate email to prevent path traversal in Secrets Manager lookup
    if not email or "/" in email or ".." in email:
        return _error_response(401, "Invalid user identity in token claims.")

    logger.info("[BasicAuth] User identity resolved from onBehalfOf claim.")

    credentials = get_basic_auth_credentials(email)
    if not credentials:
        return _error_response(401, "No credentials found for user.")

    basic_auth_encoded = base64.b64encode(
        f"{credentials['username']}:{credentials['password']}".encode()
    ).decode()
    headers["Authorization"] = f"Basic {basic_auth_encoded}"

    logger.info(f"[BasicAuth] Token translated — role: {credentials.get('role', 'unknown')}.")
    return headers


# ===========================================================================
# Pattern 2: HMAC-SHA256 signature generation
# ===========================================================================

def build_hmac_signature_headers(headers):
    """Generate HMAC-SHA256 signature over method, path, timestamp, and
    gateway host. Injects X-Signature, X-Timestamp, X-Gateway-Host headers.
    Removes the original Authorization header."""
    logger.info("[HMAC] Building HMAC signature headers.")

    headers.pop("Authorization", None)

    try:
        hmac_secret = get_hmac_secret()
    except (ValueError, ConnectionError, RuntimeError) as e:
        logger.error(f"[HMAC] Cannot build signature: {e}")
        return _error_response(500, "HMAC secret not available.")

    gateway_host = headers.get("Host", "")
    method = os.environ.get("HMAC_METHOD", "GET")    # Configure via environment variable
    path = os.environ.get("HMAC_PATH", "/image")     # Configure via environment variable
    timestamp = str(int(time.time()))
    message = f"{method}:{path}:{timestamp}:{gateway_host}"

    signature = hmac.new(
        hmac_secret.encode(), message.encode(), hashlib.sha256
    ).hexdigest()

    headers["X-Signature"] = signature
    headers["X-Timestamp"] = timestamp
    headers["X-Gateway-Host"] = gateway_host

    logger.info(f"[HMAC] Signature generated (ts: {timestamp}, gateway: {gateway_host}).")
    return headers


# ===========================================================================
# Response envelope helpers
# ===========================================================================

def _transformed_request(headers, body):
    return {"interceptorOutputVersion": "1.0", "mcp": {"transformedGatewayRequest": {"headers": headers, "body": body}}}

def _passthrough(gw_req):
    return {"interceptorOutputVersion": "1.0", "mcp": {"transformedGatewayRequest": {"headers": gw_req.get("headers", {}), "body": gw_req.get("body", {})}}}

def _error_response(status_code, message):
    return {"interceptorOutputVersion": "1.0", "mcp": {"transformedGatewayResponse": {"statusCode": status_code, "headers": {}, "body": {"error": message}}}}


# ===========================================================================
# Main handler — tool-specific authentication routing
# ===========================================================================

def lambda_handler(event, context):
    """Route requests to the appropriate auth handler based on tool name."""
    logger.info("Multi-Tool Interceptor invoked.")

    if "mcp" not in event:
        return _error_response(400, "Invalid request structure.")

    # Response interception — passthrough
    if event["mcp"].get("gatewayResponse") is not None:
        gw_resp = event["mcp"]["gatewayResponse"]
        return {"interceptorOutputVersion": "1.0", "mcp": {"transformedGatewayResponse": {
            "statusCode": gw_resp.get("statusCode", 200),
            "headers": gw_resp.get("headers", {}),
            "body": gw_resp.get("body", {})
        }}}

    # Request interception — route by tool name
    gw_req = event["mcp"]["gatewayRequest"]
    headers = gw_req.get("headers", {}).copy()
    body = gw_req.get("body", {})
    tool_name = body.get("params", {}).get("name", "")
    logger.info(f"[Request] Tool: {tool_name}")

    # Pattern 1: JWT-to-Basic-Auth
    if "basic-auth-tool" in tool_name.lower() or "get_user_data" in tool_name.lower():  # Replace with your tool name pattern
        result = build_basic_auth_header(headers)
        if isinstance(result, dict) and "mcp" in result:
            return result
        return _transformed_request(result, body)

    # Pattern 2: HMAC signing
    elif "hmac-tool" in tool_name.lower() or "hmac-image-api" in tool_name.lower() or "fetch_hmac_image" in tool_name.lower():  # Replace with your tool name pattern
        headers = build_hmac_signature_headers(headers)
        return _transformed_request(headers, body)

    # No matching pattern — passthrough
    else:
        logger.info(f"No handler for tool: {tool_name} — passing through.")
        return _passthrough(gw_req)
'''

# ---------------------------------------------------------------------------
# Create IAM role for the Interceptor Lambda
# ---------------------------------------------------------------------------
interceptor_role_name = f'InterceptorLambdaRole-{timestamp}'
interceptor_role_resp = iam_client.create_role(
    RoleName=interceptor_role_name,
    AssumeRolePolicyDocument=json.dumps({
        'Version': '2012-10-17',
        'Statement': [{'Effect': 'Allow', 'Principal': {'Service': 'lambda.amazonaws.com'}, 'Action': 'sts:AssumeRole'}]
    })
)
interceptor_role_arn = interceptor_role_resp['Role']['Arn']

iam_client.attach_role_policy(
    RoleName=interceptor_role_name,
    PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
)

# Secrets Manager access — scope to your specific secret paths in production
iam_client.put_role_policy(
    RoleName=interceptor_role_name, PolicyName='SecretsAccess',
    PolicyDocument=json.dumps({
        'Version': '2012-10-17',
        'Statement': [{
            'Effect': 'Allow',
            'Action': ['secretsmanager:GetSecretValue'],
            'Resource': [f'arn:aws:secretsmanager:{region}:{account_id}:secret:basic-auth-demo/{user_pool_id}/*', f'arn:aws:secretsmanager:{region}:{account_id}:secret:{hmac_secret_name}*']
        }]
    })
)

print('Waiting for IAM role propagation...')
time.sleep(10)

# ---------------------------------------------------------------------------
# Package and deploy the Lambda function
# ---------------------------------------------------------------------------
zip_buf = io.BytesIO()
with zipfile.ZipFile(zip_buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('lambda_function.py', interceptor_code)
zip_buf.seek(0)

interceptor_fn_name = f'CustomAuthInterceptor-{timestamp}'
interceptor_resp = lambda_client.create_function(
    FunctionName=interceptor_fn_name,
    Runtime='python3.12',
    Role=interceptor_role_arn,
    Handler='lambda_function.lambda_handler',
    Code={'ZipFile': zip_buf.read()},
    Environment={'Variables': {
        'USER_POOL_ID': user_pool_id,
        'HMAC_SECRET_NAME': hmac_secret_name,
        'HMAC_METHOD': 'GET',       # Replace with your downstream tool's HTTP method
        'HMAC_PATH': '/image'       # Replace with your downstream tool's API path
    }},
    Timeout=30,
    MemorySize=256
)
interceptor_arn = interceptor_resp['FunctionArn']
print(f'Interceptor Lambda created: {interceptor_arn}')

# Add resource policy to restrict invocation to the gateway only
lambda_client.add_permission(
    FunctionName=interceptor_fn_name,
    StatementId='AllowGatewayInvoke',
    Action='lambda:InvokeFunction',
    Principal='bedrock-agentcore.amazonaws.com',
    SourceArn=f'arn:aws:bedrock-agentcore:{region}:{account_id}:gateway/{gateway_id}'
)
print('Lambda resource policy added: restricted to gateway invocation only')

# Wait for Active state
for _ in range(12):
    time.sleep(5)
    state = lambda_client.get_function(FunctionName=interceptor_fn_name)['Configuration']['State']
    if state == 'Active':
        print('Interceptor Lambda is active.')
        break
else:
    print(f'Warning: Lambda state is {state}')


## Step 4: Attach Interceptor to Gateway and Configure Target

This step attaches the interceptor to your gateway and creates a gateway target with the required `metadataConfiguration`.

**Attaching the interceptor:** You must enable `passRequestHeaders`. Without it, the interceptor cannot receive the `Authorization` header containing the inbound JWT token, and the authentication patterns will not work.

**Configuring `allowedRequestHeaders`:** If your interceptor adds custom headers, you must include them in the target's `metadataConfiguration.allowedRequestHeaders`. Otherwise, the gateway strips them before forwarding. For the HMAC pattern in this tutorial:
- `X-Signature`, `X-Timestamp`, `X-Gateway-Host`

**Why a placeholder API key:** The gateway requires a credential provider for each target. Since the interceptor handles the real authentication, use API key with a placeholder value. The downstream tool should safely ignore the `X-API-Key` header.

In [ ]:
# ---------------------------------------------------------------------------
# Attach interceptor to gateway
# ---------------------------------------------------------------------------
agentcore_client.update_gateway(
    gatewayIdentifier=gateway_id,
    interceptorConfigurations=[
        {
            'interceptor': {
                'lambda': {
                    'arn': interceptor_arn
                }
            },
            'interceptionPoints': ['REQUEST'],
            'inputConfiguration': {
                'passRequestHeaders': True
            }
        }
    ]
)

print('Waiting for gateway update...')
for _ in range(30):
    time.sleep(10)
    status = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)['status']
    print(f'  Status: {status}')
    if status == 'READY':
        print('Interceptor attached successfully.')
        break

# ---------------------------------------------------------------------------
# Create API Key credential provider (placeholder)
# ---------------------------------------------------------------------------
api_key_response = agentcore_client.create_api_key_credential_provider(
    name=f'interceptor-placeholder-key-{timestamp}',
    apiKey='placeholder-not-used'
)
provider_arn = api_key_response['credentialProviderArn']
print(f'\nAPI Key provider created: {provider_arn}')

# ---------------------------------------------------------------------------
# Create Gateway Target with metadataConfiguration
# ---------------------------------------------------------------------------
# Replace <your-api-endpoint> with your downstream tool API endpoint.
# If you do not have a downstream API, you can use a mock Lambda function
# with an API Gateway endpoint for testing purposes.
openapi_spec = {
    'openapi': '3.0.0',
    'info': {'title': 'Enterprise Tool API', 'version': '1.0.0'},
    'servers': [{'url': 'https://<your-api-endpoint>.execute-api.<region>.amazonaws.com/prod'}],
    'paths': {
        '/data': {
            'post': {
                'summary': 'Get user data (Basic Auth)',
                'operationId': 'get_user_data',
                'responses': {'200': {'description': 'Success', 'content': {'application/json': {'schema': {'type': 'object'}}}}}
            }
        },
        '/image': {
            'get': {
                'summary': 'Fetch image metadata (HMAC)',
                'operationId': 'fetch_hmac_image',
                'responses': {'200': {'description': 'Success', 'content': {'application/json': {'schema': {'type': 'object'}}}}}
            }
        }
    }
}

target_response = agentcore_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name=f'enterprise-tool-{timestamp}',
    description='Enterprise tool with interceptor-managed authentication',
    targetConfiguration={
        'mcp': {
            'openApiSchema': {
                'inlinePayload': json.dumps(openapi_spec)
            }
        }
    },
    credentialProviderConfigurations=[{
        'credentialProviderType': 'API_KEY',
        'credentialProvider': {
            'apiKeyCredentialProvider': {
                'providerArn': provider_arn,
                'credentialLocation': 'HEADER',
                'credentialParameterName': 'X-API-Key'
            }
        }
    }],
    # KEY FEATURE: metadataConfiguration for custom header passthrough
    metadataConfiguration={
        'allowedRequestHeaders': [
            'X-Signature',         # For HMAC pattern
            'X-Timestamp',         # For HMAC pattern
            'X-Gateway-Host'       # For HMAC pattern
        ]
    }
)

target_id = target_response['targetId']
print(f'Gateway target created: {target_id}')
print(f'\nmetadataConfiguration:')
print(f"  Allowed Request Headers: {target_response['metadataConfiguration']['allowedRequestHeaders']}")

# Wait for target to be ready
for _ in range(30):
    time.sleep(10)
    t = agentcore_client.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
    if t['status'] == 'READY':
        print('Target is ready.')
        break

## Step 5: Verify Authentication Patterns

To verify the interceptor is working correctly:

1. **Basic Auth pattern** — Invoke the gateway with a valid JWT token containing an `onBehalfOf` claim. Check CloudWatch Logs for the interceptor Lambda to confirm `[BasicAuth] Token translated` appears in the log output.

2. **HMAC pattern** — Invoke the gateway targeting the HMAC tool. Check CloudWatch Logs for `[HMAC] Signature generated` and verify the downstream tool receives `X-Signature`, `X-Timestamp`, and `X-Gateway-Host` headers.

If the interceptor encounters errors, the logs will show the specific failure point (e.g., `[JWT] Failed to decode token`, `[BasicAuth] No credentials found for user`).

## Summary and Resource Information

### Resources

* [Gateway Request Interceptor — AWS Documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-interceptors.html)
* [AWS Secrets Manager — Developer Guide](https://docs.aws.amazon.com/secretsmanager/latest/userguide/intro.html)
* [Header Propagation with Gateway — AWS Documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-headers.html)

In [ ]:
print('=' * 80)
print('RESOURCE SUMMARY')
print('=' * 80)
print(f'\nInterceptor Lambda:  {interceptor_arn}')
print(f'Interceptor Role:    {interceptor_role_arn}')
print(f'Gateway ID:          {gateway_id}')
print(f'Target ID:           {target_id}')
print(f'API Key Provider:    {provider_arn}')
print('\n' + '=' * 80)

## Cleanup

Delete the resources created during this tutorial to avoid ongoing charges. Run the cells below in order.

In [ ]:
print('Starting cleanup...\n')

# 1. Delete Gateway Target
try:
    agentcore_client.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
    print(f'Deleted target: {target_id}')
except ClientError as e:
    print(f'Target: {e}')
time.sleep(10)

# 2. Remove interceptor from gateway (update with empty config)
try:
    agentcore_client.update_gateway(
        gatewayIdentifier=gateway_id,
        interceptorConfigurations=[]
    )
    print('Interceptor detached from gateway')
except ClientError as e:
    print(f'Gateway update: {e}')
time.sleep(10)

# 3. Delete Interceptor Lambda
try:
    lambda_client.delete_function(FunctionName=interceptor_fn_name)
    print(f'Deleted Lambda: {interceptor_fn_name}')
except ClientError as e:
    print(f'Lambda: {e}')

# 4. Delete IAM Role
try:
    iam_client.delete_role_policy(RoleName=interceptor_role_name, PolicyName='SecretsAccess')
    iam_client.detach_role_policy(RoleName=interceptor_role_name,
        PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole')
    iam_client.delete_role(RoleName=interceptor_role_name)
    print(f'Deleted role: {interceptor_role_name}')
except ClientError as e:
    print(f'IAM: {e}')

# 5. Delete API Key provider
try:
    agentcore_client.delete_api_key_credential_provider(
        credentialProviderArn=provider_arn
    )
    print(f'Deleted API Key provider')
except ClientError as e:
    print(f'API Key: {e}')

print('\nCleanup completed!')

## Conclusion

In this tutorial, you implemented two custom authentication patterns using a single Request Lambda Interceptor:

- **JWT-to-Basic-Auth credential translation** — dynamically resolving user identity from JWT claims and mapping it to per-user credentials in Secrets Manager
- **HMAC-SHA256 request signing** — generating cryptographically signed requests with replay protection and gateway origin verification

